# 🧠 Building a Language Model From Scratch

This notebook walks through every step of building and training a GPT-style language model.
We'll cover:

1. **Tokenization** — How text becomes numbers
2. **Model Architecture** — The transformer, piece by piece
3. **Training** — How the model learns
4. **Generation** — Making the model write text

---

In [ ]:
import sys
sys.path.insert(0, '..')  # So we can import from the project root

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Tokenization: From Text to Numbers

Neural networks can't process text directly — they need numbers. A **tokenizer** converts
text into a sequence of integer IDs, and back.

We use **BPE (Byte-Pair Encoding)**, which creates a vocabulary of sub-word tokens.
Common words like "the" get a single token, while rare words are split into pieces.

In [ ]:
from train_tokenizer import load_tokenizer

tokenizer = load_tokenizer()
print(f"Vocabulary size: {tokenizer.get_vocab_size()}")

# Let's see how tokenization works
text = "Once upon a time, there was a little girl named Lily."
encoded = tokenizer.encode(text)

print(f"\nOriginal text: {text}")
print(f"Tokens:        {encoded.tokens}")
print(f"Token IDs:     {encoded.ids}")
print(f"\nDecoded back:  {tokenizer.decode(encoded.ids)}")

In [ ]:
# Let's see how BPE handles unknown/rare words
rare_text = "The brontosaurus ate phenomenally"
encoded_rare = tokenizer.encode(rare_text)
print(f"Text:   {rare_text}")
print(f"Tokens: {encoded_rare.tokens}")
print("\nNotice how rare words are split into sub-word pieces!")

## 2. The Transformer Architecture

Let's build the model piece by piece to understand each component.

### 2.1 Token + Position Embeddings

Each token ID gets mapped to a learned vector. We also add positional information
since transformers don't inherently know about token ordering.

In [ ]:
from config import ModelConfig
cfg = ModelConfig()
print(f"Model config: {cfg}")

# Token embedding
token_emb = torch.nn.Embedding(cfg.vocab_size, cfg.d_model)

# Let's embed a short sequence
sample_ids = torch.tensor([[42, 100, 200, 5]])  # batch=1, seq=4
embedded = token_emb(sample_ids)
print(f"\nInput shape:    {sample_ids.shape}  (batch, seq_len)")
print(f"Embedded shape: {embedded.shape}  (batch, seq_len, d_model)")
print(f"\nEach token ID becomes a {cfg.d_model}-dimensional vector!")

### 2.2 Self-Attention (The Key Innovation)

Self-attention lets each token look at all other tokens to gather context.
The attention score between token i and token j tells us "how much should token i
pay attention to token j?"

For **causal** (autoregressive) models, we mask out future positions so each
token can only attend to previous tokens.

In [ ]:
import math

# Let's compute attention step by step
seq_len = 6
d_model = cfg.d_model
d_head = d_model // cfg.n_heads  # Dimension per attention head

# Dummy Q, K, V (normally these come from linear projections)
torch.manual_seed(42)
Q = torch.randn(1, seq_len, d_head)  # Query
K = torch.randn(1, seq_len, d_head)  # Key
V = torch.randn(1, seq_len, d_head)  # Value

# Step 1: Compute attention scores (Q·K^T / √d_k)
attn_scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_head)
print(f"Attention scores shape: {attn_scores.shape} (seq_len × seq_len)")

# Step 2: Apply causal mask (upper triangle → -inf)
causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
attn_scores_masked = attn_scores.masked_fill(causal_mask, float('-inf'))

# Step 3: Softmax → attention weights
attn_weights = F.softmax(attn_scores_masked, dim=-1)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(attn_scores[0].detach(), cmap='RdBu_r')
axes[0].set_title('Raw Attention Scores')
axes[0].set_xlabel('Key position'); axes[0].set_ylabel('Query position')

axes[1].imshow(attn_scores_masked[0].detach(), cmap='RdBu_r')
axes[1].set_title('After Causal Mask')
axes[1].set_xlabel('Key position'); axes[1].set_ylabel('Query position')

im = axes[2].imshow(attn_weights[0].detach(), cmap='Blues')
axes[2].set_title('Attention Weights (softmax)')
axes[2].set_xlabel('Key position'); axes[2].set_ylabel('Query position')
plt.colorbar(im, ax=axes[2])

plt.tight_layout()
plt.savefig('../checkpoints/attention_demo.png', dpi=100)
plt.show()
print("\nNotice: each row only has non-zero weights for positions ≤ its own position!")
print("This is the CAUSAL mask — preventing information leakage from the future.")

### 2.3 The Full Model

Now let's load our full SmallGPT model and examine it.

In [ ]:
from model import SmallGPT

model = SmallGPT(cfg)

# Let's count parameters by component
print("Parameter breakdown:")
print(f"  Token embeddings:  {model.token_emb.weight.numel():>10,}")
print(f"  Position embeddings: {model.pos_emb.embedding.weight.numel():>8,}")
for i, block in enumerate(model.blocks):
    block_params = sum(p.numel() for p in block.parameters())
    print(f"  Transformer block {i}: {block_params:>9,}")
print(f"  Final LayerNorm:   {sum(p.numel() for p in model.ln_final.parameters()):>10,}")
print(f"  LM Head (tied):    (shared with token embeddings)")
print(f"  {'─'*35}")
print(f"  Total:             {model.count_parameters():>10,}")

In [ ]:
# Forward pass demo
sample_text = "The cat sat on"
encoded = tokenizer.encode(sample_text)
input_ids = torch.tensor([encoded.ids])

print(f"Input: '{sample_text}'")
print(f"Token IDs: {encoded.ids}")
print(f"Tokens: {encoded.tokens}")

logits, _ = model(input_ids)
print(f"\nLogits shape: {logits.shape}  (batch, seq_len, vocab_size)")

# What does the (untrained) model predict for the next token?
last_logits = logits[0, -1, :]  # Logits for last position
probs = F.softmax(last_logits, dim=-1)
top5 = torch.topk(probs, 5)

print(f"\nTop 5 predictions for next token (untrained, so random):")
for prob, idx in zip(top5.values, top5.indices):
    token = tokenizer.id_to_token(idx.item())
    print(f"  '{token}' — {prob.item():.4f}")

## 3. Training

Training a language model is conceptually simple:

1. Feed in a sequence of tokens: `[t₀, t₁, t₂, ..., tₙ₋₁]`
2. The model predicts the next token at each position
3. Compare predictions to actual next tokens: `[t₁, t₂, t₃, ..., tₙ]`
4. Compute cross-entropy loss and backpropagate

Run the full training with:
```bash
python train.py
```

Let's look at the training loss if available:

In [ ]:
import json, os

history_path = '../checkpoints/loss_history.json'
if os.path.exists(history_path):
    with open(history_path) as f:
        history = json.load(f)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    
    if history.get('train'):
        steps = [e['step'] for e in history['train']]
        losses = [e['loss'] for e in history['train']]
        ax.plot(steps, losses, label='Train Loss', alpha=0.7)
    
    if history.get('val'):
        steps = [e['step'] for e in history['val']]
        losses = [e['loss'] for e in history['val']]
        ax.plot(steps, losses, label='Val Loss', marker='o')
    
    ax.set_xlabel('Step'); ax.set_ylabel('Loss')
    ax.set_title('Training Progress')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.show()
else:
    print("No training history found. Run 'python train.py' first!")

## 4. Text Generation

Generation is **autoregressive**: we predict one token at a time, append it, and repeat.

Let's try different sampling strategies:

In [ ]:
from generate import load_model, generate
from config import GenerationConfig

# Load the trained model (or untrained if no checkpoint exists)
model, tokenizer = load_model()
device = next(model.parameters()).device

In [ ]:
prompt = "Once upon a time"

# Greedy decoding
print("=" * 60)
print("GREEDY DECODING (deterministic, often repetitive)")
print("=" * 60)
cfg_greedy = GenerationConfig(greedy=True, max_new_tokens=100)
print(generate(model, tokenizer, prompt, cfg_greedy, device))

print()

# Temperature sampling
for temp in [0.3, 0.8, 1.5]:
    print("=" * 60)
    print(f"TEMPERATURE = {temp}")
    print("=" * 60)
    cfg_temp = GenerationConfig(temperature=temp, top_k=50, top_p=0.9, max_new_tokens=100)
    print(generate(model, tokenizer, prompt, cfg_temp, device))
    print()

In [ ]:
# Let's visualize the effect of temperature on the probability distribution
encoded = tokenizer.encode("The little dog")
input_ids = torch.tensor([encoded.ids], device=device)

with torch.no_grad():
    logits, _ = model(input_ids)

last_logits = logits[0, -1, :]  # Next-token logits

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
temps = [0.3, 1.0, 2.0]

for ax, temp in zip(axes, temps):
    probs = F.softmax(last_logits / temp, dim=-1)
    top_k = 20
    top_probs, top_indices = torch.topk(probs, top_k)
    tokens = [tokenizer.id_to_token(i.item()) for i in top_indices]
    
    ax.barh(range(top_k), top_probs.cpu().numpy())
    ax.set_yticks(range(top_k))
    ax.set_yticklabels(tokens, fontsize=8)
    ax.set_title(f'Temperature = {temp}')
    ax.set_xlabel('Probability')
    ax.invert_yaxis()

plt.suptitle('Effect of Temperature on Next-Token Probabilities', fontsize=13)
plt.tight_layout()
plt.savefig('../checkpoints/temperature_demo.png', dpi=100)
plt.show()
print("Low temp → peaked (confident). High temp → flat (random).")

## 5. What's Next?

Now that you understand the basics, here are some directions to explore:

1. **Scale up**: Increase model size (`n_layers`, `d_model`) and train longer
2. **Try different datasets**: Wikipedia, code, poetry, etc.
3. **Add features**: Rotary positional embeddings (RoPE), KV-cache for faster generation
4. **Fine-tuning**: Take a pre-trained model and fine-tune on a specific task
5. **RLHF**: Add reinforcement learning from human feedback (how ChatGPT is trained)

Happy learning! 🚀